# Machine Learning Exercise: Heart Disease from PostgreSQL

Apply the workflow from notebook 1 to data stored across five PostgreSQL tables.
Inspect the schema, build the join, check the target, then compare classifiers
without using the test set for selection.

The database and SQL work are part of the exercise. Keep individual missing
measurements for imputation, and explain your data-quality choices.
The worked solution follows the same workflow; it is not a clinically
validated model.

## Task

1. Inspect the five tables in the `heart` schema and join them using the patient key.
2. Require an existing record in each measurement table, check one row per patient,
   and handle repeated input-and-label profiles before splitting.
3. Preserve `heart_attack` and derive `disease_present = 1 - heart_attack`.
   Exclude both target columns, IDs, and source tags from the inputs.
4. Reserve stratified test data before EDA; explore training data only.
5. Build a pipeline that imputes, encodes, scales, and fits logistic regression.
   Fit preprocessing separately inside each CV training fold.
6. Compare a baseline, scalers, and hyperparameters, then freeze one approach
   and evaluate it once on test data.

**Deliver:** the SQL join, quality decisions, validation comparison,
final classification report and labelled confusion matrix, and a short
explanation of the result and its limitations.

## Keep in Mind

- Choose the positive class and success metric before tuning. Accuracy alone may hide important mistakes.
- Use identical folds and the same metric to compare candidates.
- Keep missing feature values for an imputer instead of silently removing difficult test cases.
- Numerical category codes are not necessarily continuous measurements.
- Tune with a modest randomized or grid search before increasing the search budget.
- Do not infer clinical meaning or data provenance from correlations alone.

## Data Overview

This is a **teaching adaptation of the UCI Cleveland Heart Disease dataset**,
split across five PostgreSQL tables. One joined row represents a patient record.
The task is retrospective classification of **recorded disease**, not predicting
a future heart attack. The data is small and historical; this is not a medical
decision tool. [Original UCI documentation](https://archive.ics.uci.edu/dataset/45/heart+disease).

### Target: Do Not Trust the Column Name

The database's `heart_attack` field has **reversed binary coding** compared
with the original UCI disease-present label:

| Database `heart_attack` | Derived `disease_present` | Meaning |
|---|---|---|
| 1 | 0 | No disease recorded under the source definition |
| 0 | 1 | Disease present |

Preserve the original field, check that it contains only 0 and 1, and derive
`disease_present = 1 - heart_attack`. Use **only `disease_present` as the
modelling target**. Class 0 does not mean generally healthy.
Neither target column belongs in the input features.

This mapping was checked against UCI on 2026-10-08: all 303 records with the
required joined tables agree after accounting for category recoding and missing
measurements. A higher heart rate or less angina in database class 1 therefore
does not indicate a reversed clinical relationship: that class means
**no disease recorded**. Use the documented coding, not correlations, to
interpret labels.

### Feature Dictionary: Database Codes, Not Original UCI Codes

An ECG records the heart's electrical activity. Angina refers to chest pain
associated with reduced blood supply to the heart; the ST segment is part of
the ECG. Numerical codes below are category identifiers, not measurements.

| Database column | Meaning and coding | Treatment |
|---|---|---|
| `patient_id` | Patient key from the join | Check uniqueness; exclude from inputs |
| `age` | Age in years | Numerical |
| `sex` | Recorded sex: 0 = female, 1 = male | Categorical |
| `chest_pain_type` | 0 = asymptomatic, 1 = atypical angina, 2 = non-anginal pain, 3 = typical angina | Categorical |
| `resting_blood_pressure` | Resting blood pressure on admission, mm Hg | Numerical |
| `serum_cholestoral` | Serum cholesterol, mg/dL; spelling follows the schema | Numerical |
| `fasting_blood_sugar` | Above 120 mg/dL: 0 = false, 1 = true | Categorical |
| `resting_electrocardiographic_results` | 0 = probable/definite left ventricular hypertrophy, 1 = normal, 2 = ST-T abnormality | Categorical |
| `maximum_heartrate_achieved` | Maximum exercise heart rate, beats/minute | Numerical |
| `exercise_induced_angina` | 0 = no, 1 = yes | Categorical |
| `oldpeak` | ST depression induced by exercise relative to rest | Numerical |
| `slope_of_the_peak_exercise_st_segment` | 0 = downsloping, 1 = flat, 2 = upsloping | Categorical |
| `number_of_major_vessels_colored_by_flourosopy` | Vessel count 0 to 3; code 4 represents missing | Numerical count; replace 4 with missing |
| `thal` | Test result: 1 = fixed defect, 2 = normal, 3 = reversible defect; code 0 represents missing | Categorical; replace 0 with missing |
| `real_data` | Teaching source tag, sometimes absent | Quality check only; exclude from inputs |
| `heart_attack` | Original, inverted binary field | Preserve for checking; exclude from inputs |
| `disease_present` | Derived label: 0 = no disease recorded, 1 = disease present | Target only |

The database codes differ from the original UCI codes; **do not mix the two
dictionaries**. Keep the database categories as they are and one-hot encode
them. Convert the two missing-value sentinels before imputation.

Why these 13 inputs? They retain the source's diagnostic benchmark attributes.
Vessel counts and `thal` depend on specialized tests, and several other features
come from an exercise test. They would not all exist at an earlier screening
moment. Choosing inputs requires knowing what is available when a prediction
will be made, not simply using every column in the database.

### What Is Different from the Original File?

The audit found 303 joined records: 264 match all 13 UCI measurements after
recoding, and 39 match the available measurements but have extra missing
age, sex, or maximum-heart-rate values. One original input-and-label profile
is repeated under two IDs, while another original record is absent.

The former left join also retained 50 extra patients without the required
measurement-table records. Distinguish **an absent joined record** from
**a missing measurement in an existing record**: exclude the former, but retain
the latter for training-fitted imputation. After removing the repeated profile,
the current solution uses 302 records. These counts describe the audited
teaching database, not a guaranteed future database state.

## Import

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

# Add modeling imports as you use them.

## Getting the Data

The `heart` schema contains `patient`, `chest_pain`, `blood_metrics`,
`pressure_vessels_angina`, and `heart_attack_history`.
Inspect them in DBeaver, then write the join yourself.
Use `p.id = other_table.patient_id`, select explicit columns, and alias the
patient key as `patient_id`.

Use inner joins for the measurement tables to require an existing record,
rather than keeping patients with no measurements at all. An inner join does
**not** remove null values inside an existing row: keep those for imputation.
Exclude the explicitly fabricated source tag shown in the worked solution;
a missing source tag by itself is not a reason to discard a record.

Copy `.env.example` to `.env` and fill in the supplied credentials.
The code builds the connection URL from those fields, so enter the password
as-is, without URL encoding. A complete existing `DB_STRING` is also supported.
Never commit credentials.

The next query is a **ten-patient preview**, not a modelling dataset.
Replace it with your complete five-table join before continuing.
The connection is read-only and closed after reading; the DataFrame remains available.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "solutions":
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

DB_STRING = os.getenv("DB_STRING")
if not DB_STRING or "<" in DB_STRING:
    # Build the URL from the fields in .env; URL.create handles special characters.
    field_names = ("USER_DB", "PASSWORD", "HOST", "DATABASE")
    fields = {name: os.getenv(name) for name in field_names}
    if any(not value or "<" in value for value in fields.values()):
        raise ValueError(
            "Set DB_STRING or USER_DB, PASSWORD, HOST, DATABASE and PORT in .env."
        )
    try:
        port = int(os.getenv("PORT") or "5432")
    except ValueError:
        raise ValueError("PORT in .env must be an integer.") from None
    DB_STRING = URL.create(
        "postgresql+psycopg2",
        username=fields["USER_DB"],
        password=fields["PASSWORD"],
        host=fields["HOST"],
        port=port,
        database=fields["DATABASE"],
    )

db = create_engine(DB_STRING, connect_args={"connect_timeout": 10})

In [ ]:
# Preview only: replace with your five-table join before modelling.
query_string = "SELECT * FROM heart.patient LIMIT 10"
try:
    with db.connect() as connection, connection.begin():
        connection.execute(text("SET TRANSACTION READ ONLY"))
        df_sqlalchemy = pd.read_sql(text(query_string), connection)
finally:
    db.dispose()
df_sqlalchemy.head()

In [ ]:
# Optional: save the completed join locally to the Git-ignored file.
# df_sqlalchemy.to_csv(PROJECT_ROOT / "data" / "heart_data.csv", index=False)

## 1. Check Your Completed Join

Replace the preview with your join. Check row counts, missing or repeated
patient keys, and target codes. Do not silently discard conflicting rows
for the same patient: fix the join.

Count exact duplicate **input-and-label profiles**, excluding `patient_id`
and `real_data` from this comparison. The current database has one repeated
profile under different IDs. Remove that repeated profile before splitting
so the same complete observation cannot appear on both sides of evaluation.
Identical profiles do not prove that two records identify the same real person;
this is a documented teaching-data cleanup.

Convert vessel-count code 4 and `thal` code 0 to missing values. Do not use
these sentinel codes as measurements or categories. Retain ordinary missing
inputs for the pipeline's imputer.

**Expected result:** a joined DataFrame with one row per eligible patient,
a report of exclusions, and both the original and correctly derived target.
Currently, the join gives 303 rows and profile deduplication leaves 302;
inspect your actual result rather than forcing those counts.

In [ ]:
# Inspect df_sqlalchemy and check the join before creating your modeling DataFrame.
# Explain exclusions; retain patient_id for checks, not as a model feature.

## 2. Define the Target, Metric, and Split

Preserve `heart_attack`, validate its binary codes, and derive
`disease_present = 1 - heart_attack`. **Class 1 now means disease present.**
Exclude `heart_attack`, `disease_present`, `patient_id`, and `real_data`
from `X`; use only `disease_present` as `y`.

Choose F2 for class 1 as the main selection metric, alongside precision,
recall, and accuracy. A false negative misses a disease-present record;
a false positive flags a record with no disease under the source definition.

$$F_2 = \frac{5TP}{5TP + FP + 4FN}$$

F2 emphasizes recall; the coefficient 4 does not establish real clinical costs.
An always-positive baseline has recall 1 but raises many false alarms.
Use `pos_label=1` consistently.

Reserve a stratified test set. Use training-only CV as in notebook 1, or add
a separate validation set as in the worked solution. Preprocessing belongs
inside each CV training fold.

<img src="assets/heart-evaluation-protocol.svg" width="650" style="max-width: 100%;" alt="Training-only cross-validation, a separate validation set, and one final test evaluation.">

**Expected result:** named subsets, sizes, class proportions, and an evaluation
plan. Keep test data untouched until Step 6.
[Metric reference](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.fbeta_score.html).

In [ ]:
# Check the raw binary codes, then derive disease_present = 1 - heart_attack.
# Exclude both label columns, patient_id, and real_data from features.
# Reserve test data and record the selection metric with pos_label=1.
# If using validation data, split it from development data.

## 3. Explore Training Data

Inspect types, missingness, class balance, category codes, and numerical ranges
in training data only. Plot maximum exercise heart rate by `disease_present`
and compare the proportion with exercise-induced angina in each class.
Label class 0 as "No disease recorded" and class 1 as "Disease present".

These plots address the original confusion about healthier-looking class-1
records: that was the **raw inverted target**, not the corrected target.
Interpret the actual outputs without treating a correlation as clinical evidence.

**Expected result:** a concise quality summary and numerical/categorical
feature groups, with reasons for their treatment. Do not delete an extreme
measurement just because it looks unusual.

In [ ]:
# Explore only training rows and document the preprocessing choices they justify.

## 4. Establish a Baseline and Build a Pipeline

Choose a simple baseline appropriate to your metric, then build a `ColumnTransformer` and `Pipeline` with imputation, categorical encoding, a numerical scaler, and logistic regression. For coded categories such as chest-pain type, do not assume the integers represent equally spaced measurements.

**Expected result:** a baseline and a complete pipeline evaluated using your development-data protocol. Explain what the baseline predicts; high recall alone can hide many false positives.

In [ ]:
# Add modeling imports as needed, then build the baseline and full pipeline.
# Evaluate with development data only; do not fit preprocessing before CV.

## 5. Compare and Select Candidates

Compare unscaled, standardized, and min-max-scaled pipelines with the same metric and CV folds. Run a modest search on training folds; if you have a separate validation set, use it for the final candidate comparison. Include the baseline and keep the test scores out of this table.

**Expected result:** one comparison table and a selected configuration with a short justification. Report when tuning fails to improve on a simpler candidate.

Set `verbose=1` on both searches to show that they are running.
Start with a small budget, such as 10 random candidates and 8 grid candidates.

In [ ]:
# Compare scalers and a small hyperparameter search under your chosen protocol.
# Freeze the selected recipe before looking at the test results.

## 6. Evaluate Once on the Test Set

First answer the questions at the end. Freeze the selected configuration,
refit on all development data, and evaluate once on test data.
A CV search with `refit=True` refits on the data passed to `fit`;
a separate validation subset is not included automatically.

**Expected result:** F2, precision, recall, accuracy, a classification report,
and a confusion matrix labelled "No disease recorded" / "Disease present".
Explain false positives and false negatives using `disease_present`, not
the raw database column. Do not use the result to tune again.

In [ ]:
# Refit the frozen configuration if needed, then perform the final test evaluation.
# Keep final test results separate from your development-data comparison.

## 7. Explain Your Result

Summarize whether the selected model improves on the baseline under your chosen metric, what the errors mean, and two limitations of the data or evaluation. Distinguish dataset-label prediction from future heart-attack risk.

**Expected result:** a short conclusion accompanying your reproducible notebook. After completing it, compare your choices with the [worked solution](solutions/2_exercise_machine_learning.ipynb); different valid choices need not produce identical scores.

## Check Your Understanding

Before final test evaluation, explain:

1. Why exclude `patient_id` even if it improves validation scores?
2. Why fit the imputer inside each CV training fold?
3. Why not choose models using their test scores?
4. Why preserve `heart_attack` but never pass it to the corrected model?

Explain your reasoning before consulting the worked solution.